In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

sp.init_printing(use_unicode=True)

z = sp.Symbol('z', complex=True)
z_inv = sp.Symbol('z^{-1}', complex=True)
n = sp.Symbol('n', integer=True)

print("=== LTI System Analysis (Problem 090201) ===")


# ==============================================================================
# 1. Z-TRANSFORMS OF INPUT AND OUTPUT
# ==============================================================================

a1, a2 = sp.Rational(1, 3), sp.Integer(2)

X_right = sp.simplify(1 / (1 - a1 / z))
X_left = sp.simplify(z / (a2 - z))
X_z = sp.factor(sp.cancel(sp.together(X_right + X_left)))

b1, b2 = sp.Rational(1, 3), sp.Rational(2, 3)

Y_1 = sp.simplify(5 / (1 - b1 / z))
Y_2 = sp.simplify(-5 / (1 - b2 / z))
Y_z = sp.factor(sp.cancel(sp.together(Y_1 + Y_2)))

print("\n1. Z-transform of the input:")
display(sp.Eq(sp.Symbol('X(z)'), X_z))

print("\n2. Z-transform of the output:")
display(sp.Eq(sp.Symbol('Y(z)'), Y_z))


# ==============================================================================
# 2. TRANSFER FUNCTION
# ==============================================================================

H_z = sp.factor(sp.cancel(Y_z / X_z))

print("\n3. Transfer function:")
display(sp.Eq(sp.Symbol('H(z)'), H_z))

H_z_inv = sp.factor(sp.cancel(H_z.subs(z, 1 / z_inv)))

print("\nH(z) in z^{-1} form:")
display(sp.Eq(sp.Symbol('H(z)'), H_z_inv))


# ==============================================================================
# 3. POLES AND ZEROS
# ==============================================================================

num_H, den_H = sp.fraction(H_z)

zeros_H = sp.solve(num_H, z)
poles_H = sp.solve(den_H, z)

print("\n4. Zeros:")
display(zeros_H)

print("Poles:")
display(poles_H)


# ==============================================================================
# 4. IMPULSE RESPONSE FROM TIME-SHIFT STRUCTURE
# ==============================================================================

num_inv, den_inv = sp.fraction(H_z_inv)

poly_den = sp.Poly(sp.expand(den_inv), z_inv)
c1 = poly_den.coeff_monomial(z_inv)
c0 = poly_den.coeff_monomial(1)

a = sp.simplify(-c1 / c0)
F = sp.simplify(1 / (1 - a * z_inv))

structure = sp.simplify(sp.cancel(H_z_inv / F))
poly_structure = sp.Poly(sp.expand(structure), z_inv)

A = sp.simplify(poly_structure.coeff_monomial(1))
B = sp.simplify(poly_structure.coeff_monomial(z_inv))

h_n = sp.simplify(A * a**n * sp.Heaviside(n) + B * a**(n - 1) * sp.Heaviside(n - 1))

print("\n5. Impulse response:")
display(sp.Eq(sp.Symbol('h[n]'), h_n))


# ==============================================================================
# 5. DIFFERENCE EQUATION
# ==============================================================================

num_eq, den_eq = sp.fraction(H_z_inv)

den_eq = sp.expand(den_eq)
num_eq = sp.expand(num_eq)

leading = sp.Poly(den_eq, z_inv).coeff_monomial(1)

den_eq = sp.expand(den_eq / leading)
num_eq = sp.expand(num_eq / leading)

den_poly = sp.Poly(den_eq, z_inv)
num_poly = sp.Poly(num_eq, z_inv)

y = sp.Function('y')
x = sp.Function('x')

lhs = sum(coef * y(n - power) for (power,), coef in den_poly.terms())
rhs = sum(coef * x(n - power) for (power,), coef in num_poly.terms())

lhs = sp.simplify(lhs)
rhs = sp.simplify(rhs)

print("\n6. Difference equation:")
display(sp.Eq(lhs, rhs))


# ==============================================================================
# 6. NUMERICAL FUNCTION FOR h[n]
# ==============================================================================

def evaluate_h(expr, n_values):
    values = []
    for k in n_values:
        value = expr.subs(n, int(k))
        value = value.replace(sp.Heaviside, lambda arg: sp.Integer(1) if arg >= 0 else sp.Integer(0))
        values.append(float(sp.N(value)))
    return np.array(values)


# ==============================================================================
# 7. POLE-ZERO DIAGRAM
# ==============================================================================

zero_values = [complex(sp.N(v)) for v in zeros_H]
pole_values = [complex(sp.N(v)) for v in poles_H]

theta = np.linspace(0, 2 * np.pi, 400)

fig, ax = plt.subplots(figsize=(7, 7))

ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')
ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)

if zero_values:
    ax.scatter([v.real for v in zero_values], [v.imag for v in zero_values],
               s=130, facecolors='none', edgecolors='blue', linewidths=2,
               marker='o', label='Zeros')

if pole_values:
    ax.scatter([v.real for v in pole_values], [v.imag for v in pole_values],
               s=130, color='red', marker='x', linewidths=2, label='Poles')

ax.set_aspect('equal')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_xlabel('Re{z}')
ax.set_ylabel('Im{z}')
ax.set_title('Pole-Zero Diagram of H(z)')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend()

plt.show()


# ==============================================================================
# 8. FREQUENCY RESPONSE
# ==============================================================================

omega_symbol = sp.Symbol('omega', real=True)

H_omega_expr = H_z.subs(z, sp.exp(sp.I * omega_symbol))
H_omega_func = sp.lambdify(omega_symbol, H_omega_expr, 'numpy')

omega = np.linspace(0, np.pi, 500)
H_omega = H_omega_func(omega)

magnitude_db = 20 * np.log10(np.maximum(np.abs(H_omega), 1e-12))
phase_deg = np.unwrap(np.angle(H_omega)) * 180 / np.pi

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(omega / np.pi, magnitude_db, 'b')
ax1.set_xlabel(r'Normalized Frequency ($\omega/\pi$)')
ax1.set_ylabel('Magnitude (dB)')
ax1.set_title('Frequency Response — Magnitude')
ax1.grid(True)

ax2.plot(omega / np.pi, phase_deg, 'r')
ax2.set_xlabel(r'Normalized Frequency ($\omega/\pi$)')
ax2.set_ylabel('Phase (degrees)')
ax2.set_title('Frequency Response — Phase')
ax2.grid(True)

plt.tight_layout()
plt.show()


# ==============================================================================
# 9. IMPULSE RESPONSE
# ==============================================================================

n_vec = np.arange(-5, 16)
h_values = evaluate_h(h_n, n_vec)

plt.figure(figsize=(10, 4))

plt.stem(n_vec, h_values, basefmt=" ")

plt.axhline(0, color='black', linewidth=1)
plt.xlabel('n')
plt.ylabel('h[n]')
plt.title('Impulse Response h[n]')
plt.grid(True)

plt.show()